# SupScene heatmap visualization

In [ ]:
import os, sys
import numpy as np
import cv2
import torch
import torch.nn.functional as F
from PIL import Image
import matplotlib.cm as cm
from pathlib import Path
from safetensors.torch import load_file
project_root = Path.cwd()
os.chdir(project_root.parent)

if str(project_root.parent) not in sys.path:
    sys.path.insert(0, str(project_root.parent))

from supscene.encoder import create_encoder
from supscene.viz_utils.viz_assign_map import create_overlay_batch


print("✅ import successfully")

## 1.initialization and weight loading

In [ ]:
# ===== config params====
model_configs = {
        "dinov2_salad": {
            "backbone": {
                "name": "dinov2", 
                "args": {
                    "model_name": "dinov2_vitb14", 
                    "num_trainable_blocks": 1,
                    "return_cls_token": True
                }
            },
            "aggregator": {
                "name": "salad", 
                "args": {}
            },
            "deploy_head": {
                "name": "deploy_head", 
                "args": {}
            },
            "weights": {"type": "salad"}
        },
        
        "dinov2_netvlad": {
            "backbone": {
                "name": "dinov2", 
                "args": {
                    "model_name": "dinov2_vitb14", 
                    "num_trainable_blocks": 1
                }
            },
            "aggregator": {
                "name": "netvlad", 
                "args": {"num_clusters": 64}
            },
            "deploy_head": {
                "name": "deploy_head", 
                "args": {}
            },
            "weights": {}
        },
        "dinov2_divlad":
        {
            "backbone": {
                "name": "dinov2", 
                "args": {
                    "model_name": "dinov2_vitb14", 
                    "num_trainable_blocks": 1,
                    "return_attn_maps": True
                }
            },
            "aggregator": {
                "name": "divlad", 
                "args": {}
            },
            "deploy_head": {
                "name": "deploy_head", 
                "args": {}
            }
        }

}
# Path configuration
IMAGE_PATH = "data/GL3D/589433fa9a8c0314c5ce9656/images/00000130.jpg"  # Change to your image path
WEIGHTS_PATH = "experiments/divlad/checkpoints/last/model.safetensors"  # Change to your weights file path if available
OUTPUT_PATH = "attention_overlay_result.png"  # Output file path

# Visualization parameters
ALPHA = 0.5  # Heatmap transparency
COLORMAP = 'magma'  # Colormap: 'viridis', 'plasma', 'inferno', 'magma'
IMG_SIZE = 322  # Image size
MODEL = "dinov2_divlad"
print(f"📋 Configuration:")
print(f"   Model: {MODEL}")
print(f"   Image path: {IMAGE_PATH}")
print(f"   Weights path: {WEIGHTS_PATH}")
print(f"   Output path: {OUTPUT_PATH}")
print(f"   Colormap: {COLORMAP}")
print(f"   Alpha: {ALPHA}")

## 2. loading model and preprocess image

In [ ]:
# 1. Create model
print("🔧 Building model...")
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"📱 Using device: {device}")

model = create_encoder(model_configs[f"{MODEL}"])
model = model.to(device)
model.eval()

# 2. Load weights (if available)
if WEIGHTS_PATH and os.path.exists(WEIGHTS_PATH):
    print(f"📥 Loading weights: {WEIGHTS_PATH}")
    checkpoint = load_file(WEIGHTS_PATH)
    
    # Handle different checkpoint formats
    if 'state_dict' in checkpoint:
        state_dict = checkpoint['state_dict']
    elif 'model' in checkpoint:
        state_dict = checkpoint['model']
    else:
        state_dict = checkpoint
    
    # Smart weight loading
    model_state_dict = model.state_dict()
    filtered_state_dict = {}
    
    for key, value in state_dict.items():
        if key in model_state_dict and model_state_dict[key].shape == value.shape:
            filtered_state_dict[key] = value
    
    inc = model.load_state_dict(filtered_state_dict, strict=False)
    print(f"✅ Weights loaded, {len(filtered_state_dict)} parameters matched")
    missing = getattr(inc, "missing_keys", [])
    unexpected = getattr(inc, "unexpected_keys", [])
    print(f"Loaded weights (missing={len(missing)}, unexpected={len(unexpected)})")
else:
    print("⚠️ No weights file specified, using random initialization")

print(f"✅ Model ready")
total_params = sum(p.numel() for p in model.parameters())
print(f"   Total parameters: {total_params:,}")

## 3. image process and feed forward

In [ ]:
# 3. Preprocess image
print(f"\n📸 Preprocessing image: {IMAGE_PATH}")

if not os.path.exists(IMAGE_PATH):
    print(f"❌ Image file does not exist: {IMAGE_PATH}")
    print("Please set IMAGE_PATH to a valid image file path.")
else:
    # Read image
    image = cv2.imread(IMAGE_PATH)
    image_rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
    
    # Resize
    image_resized = cv2.resize(image_rgb, (IMG_SIZE, IMG_SIZE))
    
    # Convert to tensor and normalize
    image_tensor = torch.from_numpy(image_resized).float() / 255.0
    image_tensor = image_tensor.permute(2, 0, 1)  # HWC -> CHW
    
    # ImageNet normalization
    mean = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
    std = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)
    image_tensor = (image_tensor - mean) / std
    
    # Add batch dimension and move to device
    image_tensor = image_tensor.unsqueeze(0).to(device)
    
    print(f"✅ Image preprocessing done")
    print(f"   Original image shape: {image_rgb.shape}")
    print(f"   Tensor shape: {image_tensor.shape}")
    
    # 4. Forward pass to extract aggregator return maps
    print(f"\n🔍 Extracting aggregator return maps...")
    
    with torch.no_grad():
        # Forward backbone features
        features = model.backbone(image_tensor)
        
        # Get return maps from aggregator
        try:
            # Call aggregator forward with return_maps=True
            result = model.aggregator(features, return_maps=True)
            
            if isinstance(result, tuple) and len(result) >= 2:
                vlad_descriptor, vlad_maps = result[0], result[1]
                print(f"✅ Aggregator return maps extracted: {vlad_maps.shape}")
                print(f"   VLAD descriptor shape: {vlad_descriptor.shape}")
                
                # Convert to numpy
                vlad_maps_np = vlad_maps.squeeze(0).cpu().numpy()  # [K, H, W]
                
                print(f"   Return maps numpy shape: {vlad_maps_np.shape}")
                print(f"   Value range: [{vlad_maps_np.min():.6f}, {vlad_maps_np.max():.6f}]")
                print(f"   Sum of weights per cluster: {vlad_maps_np.sum(axis=(1,2))}")
                
            else:
                print(f"⚠️ Aggregator did not return maps, result type: {type(result)}")
                # Create dummy maps for demo
                B, C, H, W = features.shape
                vlad_maps = torch.randn(B, 64, H, W).to(device)  # Assume 64 clusters
                vlad_maps = F.softmax(vlad_maps.view(B, 64, -1), dim=-1).view(B, 64, H, W)
                vlad_maps_np = vlad_maps.squeeze(0).cpu().numpy()
                print(f"✅ Dummy return maps created: {vlad_maps_np.shape}")
                
        except Exception as e:
            print(f"❌ Failed to extract return maps: {e}")
            # Create dummy maps
            B, C, H, W = features.shape
            vlad_maps = torch.randn(B, 64, H, W).to(device)
            vlad_maps = F.softmax(vlad_maps.view(B, 64, -1), dim=-1).view(B, 64, H, W)
            vlad_maps_np = vlad_maps.squeeze(0).cpu().numpy()
            print(f"✅ Dummy return maps created: {vlad_maps_np.shape}")

## 4. heatmap visualization

In [ ]:
print(f"\n🎨 Generating aggregator return maps heatmap visualization...")

if 'vlad_maps_np' in locals():
    # Prepare / resize maps to original image size
    orig_h, orig_w = image_resized.shape[:2]
    K, H_map, W_map = vlad_maps_np.shape

    # Batch prepare
    original_images_batch = image_resized.transpose(2, 0, 1)[np.newaxis, ...]  # (1, 3, H, W)
    vlad_maps_batch = vlad_maps_np[np.newaxis, ...]  # (1, K, H, W)

    print(f"   Original image batch shape: {original_images_batch.shape}")
    print(f"   Resized aggregator maps batch shape: {vlad_maps_batch.shape}")
    
    # Heatmap overlay
    overlay_result_batch = create_overlay_batch(
        original_images=original_images_batch,
        attention_maps=vlad_maps_batch,
        alpha=ALPHA,
        colormap_name=COLORMAP
    )
    overlay_image_chw = overlay_result_batch[0]
    overlay_image_hwc = overlay_image_chw.transpose(1, 2, 0)
    overlay_image_bgr = cv2.cvtColor(overlay_image_hwc, cv2.COLOR_RGB2BGR)
    cv2.imwrite(OUTPUT_PATH, overlay_image_bgr)
    
    # -------- Cluster assignment visualization (resized) --------
    if (H_map, W_map) != (orig_h, orig_w):
        print(f"   🔄 Resizing VLAD maps from ({H_map}, {W_map}) to ({orig_h}, {orig_w})")
        vlad_maps_resized_np = np.empty((K, orig_h, orig_w), dtype=np.float32)
        for k in range(K):
            vlad_maps_resized_np[k] = cv2.resize(
                vlad_maps_np[k], (orig_w, orig_h), interpolation=cv2.INTER_AREA
            )
    else:
        vlad_maps_resized_np = vlad_maps_np
        print("   ✅ VLAD maps already match target size")
    K = vlad_maps_resized_np.shape[0]
    cluster_assign = vlad_maps_resized_np.argmax(0)  # (H, W)
    
    colors = (cm.get_cmap('jet', K)(np.arange(K))[:, :3] * 255).astype(np.uint8)
    cluster_color_image = colors[cluster_assign]
    
    cluster_overlay = (
        (1 - ALPHA) * image_resized.astype(np.float32) + ALPHA * cluster_color_image.astype(np.float32)
    ).astype(np.uint8)
    
    CLUSTER_OUTPUT_PATH = "cluster_overlay_result.png"
    cv2.imwrite(CLUSTER_OUTPUT_PATH, cv2.cvtColor(cluster_overlay, cv2.COLOR_RGB2BGR))
    
    print(f"✅ Aggregator heatmap saved: {OUTPUT_PATH}")
    print(f"✅ Cluster assignment overlay saved: {CLUSTER_OUTPUT_PATH}")
    
    # Display comparison
    try:
        import matplotlib.pyplot as plt
        fig, axes = plt.subplots(1, 3, figsize=(18, 6))
        axes[0].imshow(image_resized)
        axes[0].set_title('Original')
        axes[0].axis('off')
        
        axes[1].imshow(overlay_image_hwc)
        axes[1].set_title(f'Heatmap ({COLORMAP})')
        axes[1].axis('off')
        
        axes[2].imshow(cluster_overlay)
        axes[2].set_title(f'Cluster Assign (K={K})')
        axes[2].axis('off')
        plt.tight_layout()
        plt.show()
    except Exception as e:
        print(f"⚠️ Failed to display figure: {e}")